# CCC Batch Full Run

Run full CCC scenarios through all steps and Vorblatt/Begründung export. Keep credentials in environment variables or the repo-root `.env`, not in notebook output. The runner loads `.env` automatically without overriding values already set in the shell.

A batch is a matrix:

`law_pairs × models × deep_research_modes × repetitions`

Example: 4 law pairs × 2 models × 2 DR modes × 1 repetition = 16 full sessions.

Important config fields:

- `law_pairs`: tuple of built-in law-pair names without `_gueltig.txt` / `_vorschlag.txt`. Empty tuple means all discovered built-in pairs.
- `models`: tuple of `ModelSpec(provider, model)` entries.
- `deep_research_modes`: `(True, False)` runs each scenario once with DR and once without DR.
- `repetitions`: repeat count per law/model/DR combination. Use `1` for exactly one run per combination.
- `concurrency`: how many full sessions may run in parallel. Use `1` for careful smoke tests; `4` is a reasonable bounded batch default.
- `max_run_attempts`: how often a scenario may restart run-all if a normal step fails or appears stuck. Completed steps are kept; the script does not reset/undo steps.
- `max_deep_research_attempts`: separate cap for fresh Deep Research failures. Recommended default is low because DR is expensive; later normal-step retries can still reuse a successful DR result.
- `max_stuck_minutes`: retry threshold when run-all stays running without observable progress. This is not a total runtime limit. Set to `0` to disable the stuck guard; the default is deliberately generous because single LLM/DR calls can be slow.
- `output_dir`: root folder for batch outputs. Each call to `run_batch(config)` creates a timestamped child folder unless `batch_id` is set.
- `batch_id`: optional readable folder name under `output_dir`, useful when you want to label a manual run yourself. Reuse the same `batch_id` to resume a batch.
- `resume_completed`: `True` by default. When rerunning the same `batch_id`, scenarios already logged as `success` are skipped; failed or missing scenarios run again.
- `limit_scenarios`: optional safety cap for smoke tests. Set to `None` for the full matrix.

In [ ]:
import os
import sys
from dataclasses import replace
from pathlib import Path

repo_root = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from scripts.batch_full_run import BatchConfig, ModelSpec, load_batch_result, load_dotenv_if_available, run_batch

load_dotenv_if_available(repo_root / ".env")


In [ ]:
# Smoke run: one law pair, one model, without DR.
config = BatchConfig(
    base_url="http://localhost:5000",
    email=os.environ["CCC_BATCH_EMAIL"],
    password=os.environ["CCC_BATCH_PASSWORD"],
    law_pairs=("arbeitstagepauschale",),
    models=(ModelSpec("gemini", "gemini-3.5-flash"),),
    deep_research_modes=(False,),
    repetitions=1,
    concurrency=1,
    max_run_attempts=2,
    max_deep_research_attempts=1,
    output_dir=repo_root / "batch_runs",
    batch_id="smoke_test",
    built_in_laws_dir=repo_root / "resources" / "built_in_laws",
    limit_scenarios=1,
)


In [ ]:
# One repetition for every built-in law pair × both Gemini models × with/without DR.
# Empty law_pairs means: use all discovered built-in law pairs.
config = BatchConfig(
    base_url="http://localhost:5000",
    email=os.environ["CCC_BATCH_EMAIL"],
    password=os.environ["CCC_BATCH_PASSWORD"],
    law_pairs=(),
    models=(
        ModelSpec("gemini", "gemini-3.1-pro-preview"),
        ModelSpec("gemini", "gemini-3.5-flash"),
        ModelSpec("gemini", "gemini-3.6-flash")
    ),
    deep_research_modes=(True, False),
    repetitions=1,
    concurrency=4,
    max_run_attempts=5,
    max_deep_research_attempts=1,
    max_stuck_minutes=60,
    output_dir=repo_root / "batch_runs",
    batch_id="full_matrix_once",
    built_in_laws_dir=repo_root / "resources" / "built_in_laws",
    limit_scenarios=None,
)


In [ ]:
results = await run_batch(config)


## Rerun one scenario

Copy a `scenario_id` from `df`, `diagnostics`, or `runs.jsonl`. This creates a fresh session for exactly that matrix entry and appends the new result to the same batch folder. Use this for a failed attempt or for a targeted restart without rerunning the whole matrix.

In [ ]:
scenario_id_to_rerun = "arbeitstagepauschale__gemini-3.5-flash__no-dr__01"

rerun_config = replace(
    config,
    scenario_ids=(scenario_id_to_rerun,),
    resume_completed=False,
    concurrency=1,
    limit_scenarios=None,
)

rerun_results = await run_batch(rerun_config)
rerun_results.to_dataframe(include_diagnostics=True)


In [ ]:
df = results.to_dataframe()
df


## Reload an existing batch

Use this after editing/backfilling `runs.jsonl`, or after reopening the notebook. Point `batch_dir` at the concrete batch folder containing `runs.jsonl`.

In [ ]:
batch_dir = repo_root / "batch_runs" / "full_matrix_once"

results = load_batch_result(batch_dir)
df = results.to_dataframe()
diagnostics = results.to_dataframe(include_diagnostics=True)
summary = results.to_summary_dataframe()

df


In [ ]:
diagnostics = results.to_dataframe(include_diagnostics=True)
diagnostics


In [ ]:
summary = results.to_summary_dataframe()
summary


In [ ]:
df.groupby(["model", "deep_research", "status"]).size().unstack(fill_value=0)
